# 02 — DSL Feature Engineering

Builds `[n_alts × n_terms]` feature matrices for every choice-set event.

**Strategy (vectorized, not per-event queries):**
1. Build item-level and customer×item-level lookup dicts from the processed DataFrame
2. For each event, read features from lookups — O(1) per alternative
3. Save feature arrays to `data/train_features.pkl` for the inner loop

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from tqdm.notebook import tqdm

from src.dsl import DSLStructure, LAYER1_PRIMITIVES, LAYER2_AMAZON, ALL_TERMS

DATA_DIR = Path('../data')

In [ ]:
df = pd.read_parquet(DATA_DIR / 'purchases_processed.parquet')
with open(DATA_DIR / 'train_choices.pkl', 'rb') as f:
    train_choices = pickle.load(f)

print(f'Purchases: {len(df):,}  |  Training events: {len(train_choices):,}')
print(f'Columns: {list(df.columns)}')

## Build lookup tables (O(n) once, then O(1) per event)

In [ ]:
# --- Global item stats ---
item_popularity = np.log1p(df.groupby('asin').size()).to_dict()           # {asin: log_count}
cat_avg_price   = df.groupby('category')['price'].mean().to_dict()        # {category: avg_price}

# --- Per-customer × item: last known state at time of purchase ---
# We use the *cumcount* values (already in df) as the state features.
# Key: (customer_id, asin) -> dict of feature values at last purchase
cust_item_df = (
    df.sort_values(['customer_id', 'asin', 'order_date'])
      .groupby(['customer_id', 'asin'])
      .last()
      [['routine', 'recency_days', 'novelty', 'price']]
)
cust_item_lookup = cust_item_df.to_dict('index')   # {(cid, asin): {routine, recency_days, ...}}

# --- Per-customer × category affinity ---
cust_cat_affinity = (
    df.sort_values(['customer_id', 'category', 'order_date'])
      .groupby(['customer_id', 'category'])['cat_affinity']
      .last()
      .to_dict()
)  # {(cid, category): count}

# --- Per-customer brand affinity ---
if 'brand' not in df.columns:
    df['brand'] = df['title'].fillna('').str.split().str[0].str.lower()
cust_brand_affinity = (
    df.sort_values(['customer_id', 'brand', 'order_date'])
      .groupby(['customer_id', 'brand'])
      .size()
      .to_dict()
)  # {(cid, brand): count}

# --- Per-ASIN brand ---
asin_brand = df.groupby('asin')['brand'].first().to_dict()   # {asin: brand}

# --- Per-ASIN category ---
asin_category = df.groupby('asin')['category'].first().to_dict()  # {asin: category}

print('Lookup tables built:')
print(f'  item_popularity:    {len(item_popularity):,} ASINs')
print(f'  cust_item_lookup:   {len(cust_item_lookup):,} (customer, ASIN) pairs')
print(f'  cust_cat_affinity:  {len(cust_cat_affinity):,} (customer, category) pairs')
print(f'  cust_brand_affinity:{len(cust_brand_affinity):,} (customer, brand) pairs')

## Define vectorized feature extraction

In [ ]:
def get_item_features(cid: str, asin: str, event_category: str) -> dict:
    """
    Get feature values for one alternative in a choice set.
    Uses precomputed lookups — O(1) per call.
    """
    key = (cid, asin)
    hist = cust_item_lookup.get(key, {})

    routine_val    = float(hist.get('routine', 0))
    recency_raw    = float(hist.get('recency_days', 999))
    recency_val    = 1.0 / (1.0 + recency_raw)
    novelty_val    = 1.0 if not hist else 0.0    # never bought = novel
    pop_val        = item_popularity.get(asin, 0.0)

    cat            = asin_category.get(asin, event_category)
    aff_val        = np.log1p(cust_cat_affinity.get((cid, cat), 0.0))

    avg_price      = cat_avg_price.get(cat, 10.0)
    item_price     = float(hist.get('price', avg_price))
    price_sens     = -(item_price / (avg_price + 1e-8) - 1.0)

    brand          = asin_brand.get(asin, '')
    brand_aff      = np.log1p(cust_brand_affinity.get((cid, brand), 0.0))

    return {
        'routine':           routine_val,
        'recency':           recency_val,
        'novelty':           novelty_val,
        'popularity':        pop_val,
        'affinity':          aff_val,
        'time_match':        0.0,          # no hour-of-day in dataset
        'price_sensitivity': price_sens,
        'rating_signal':     0.0,          # not in dataset
        'brand_affinity':    brand_aff,
        'price_rank':        1.0 - (item_price / (avg_price + 1e-8)),
        'delivery_speed':    0.0,          # not in dataset
        'co_purchase':       0.0,          # requires co-purchase matrix
    }


def build_feature_matrix(event: dict, structure: DSLStructure) -> np.ndarray:
    """Build [n_alts, n_terms] feature matrix for one choice event."""
    cid   = event['customer_id']
    cat   = event['category']
    n_alt = len(event['choice_asins'])
    n_t   = len(structure.terms)

    mat = np.zeros((n_alt, n_t))
    for k, asin in enumerate(event['choice_asins']):
        feat = get_item_features(cid, asin, cat)
        for j, term in enumerate(structure.terms):
            mat[k, j] = feat.get(term, 0.0)
    return mat

print('Feature extraction functions defined.')

## Extract features for all training events

Set `MAX_EVENTS` to `None` to use the full training set.

In [ ]:
structure = DSLStructure.initial()
print(f'Structure: {structure}')

MAX_EVENTS = 10_000   # set to None for full run (~minutes on 1.85M purchases)
events = train_choices[:MAX_EVENTS] if MAX_EVENTS else train_choices

features_list  = []
chosen_indices = []
customer_ids   = []
categories     = []

for event in tqdm(events, desc='Extracting features'):
    features_list.append(build_feature_matrix(event, structure))
    chosen_indices.append(event['chosen_idx'])
    customer_ids.append(event['customer_id'])
    categories.append(event['category'])

print(f'\nDone.  Feature matrix shape per event: {features_list[0].shape}')
print(f'Terms: {structure.terms}')

## Sanity check: feature distributions for chosen items

In [ ]:
import matplotlib.pyplot as plt

chosen_feats = np.array([f[c] for f, c in zip(features_list, chosen_indices)])
print(f'Chosen-item feature matrix shape: {chosen_feats.shape}')
print(pd.DataFrame(chosen_feats, columns=structure.terms).describe().round(3))

fig, axes = plt.subplots(1, len(structure.terms), figsize=(5 * len(structure.terms), 3))
if len(structure.terms) == 1:
    axes = [axes]
for i, (term, ax) in enumerate(zip(structure.terms, axes)):
    ax.hist(chosen_feats[:, i], bins=40, color='steelblue')
    ax.set_title(term)
plt.suptitle('Feature distributions — chosen items', y=1.02)
plt.tight_layout()
plt.show()

## Sanity check: chosen vs. unchosen feature means

In [ ]:
# For each event, separate chosen from unchosen alternatives
unchosen_feats = np.vstack([
    f[np.arange(len(f)) != c]
    for f, c in zip(features_list, chosen_indices)
])

comparison = pd.DataFrame({
    'term': structure.terms,
    'chosen_mean':   chosen_feats.mean(axis=0),
    'unchosen_mean': unchosen_feats.mean(axis=0),
})
comparison['diff'] = comparison['chosen_mean'] - comparison['unchosen_mean']
print('Chosen vs. unchosen feature means:')
print(comparison.to_string(index=False))

## Save feature arrays

In [ ]:
with open(DATA_DIR / 'train_features.pkl', 'wb') as f:
    pickle.dump({
        'features_list':  features_list,
        'chosen_indices': chosen_indices,
        'customer_ids':   customer_ids,
        'categories':     categories,
        'structure':      structure.to_dict(),
    }, f)
print(f'Saved {len(features_list):,} events to data/train_features.pkl')